# London Housing Price Prediction

This notebook aims to predict housing prices in London. We will go through the steps of data loading, cleaning, feature engineering, model training, and prediction.

In [97]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## 1. Data Loading

First, we load the training, testing, and sample submission datasets into pandas DataFrames.

In [98]:
df_train = pd.read_csv('train.csv')
df_test = pd.read_csv('test.csv')
df_sub = pd.read_csv('sample_submission.csv')

## 2. Initial Data Inspection and Preprocessing

We start by inspecting the datasets for missing values and general information. We'll also drop irrelevant columns.

In [99]:
df_train.isna().sum()

,0
ID,0
fullAddress,0
postcode,0
country,0
outcode,0
latitude,0
longitude,0
bathrooms,48479
bedrooms,24843
floorAreaSqM,13806


### Dropping Irrelevant Columns

The `ID`, `fullAddress`, `country`, and `postcode` columns are either unique identifiers, contain too much specific information, or are not directly useful for our model, so we will drop them. The `outcode` column will be kept for one-hot encoding later.

In [100]:
df_train = df_train.drop(['ID', 'fullAddress', 'country', 'postcode'], axis = 1)
df_test = df_test.drop(['ID', 'fullAddress', 'country', 'postcode'], axis = 1)

### Inspecting Data Types and Non-Null Counts

We check the data types and non-null counts for both the training and test sets to understand the completeness of each feature.

In [101]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 266325 entries, 0 to 266324
Data columns (total 13 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   outcode              266325 non-null  object 
 1   latitude             266325 non-null  float64
 2   longitude            266325 non-null  float64
 3   bathrooms            217846 non-null  float64
 4   bedrooms             241482 non-null  float64
 5   floorAreaSqM         252519 non-null  float64
 6   livingRooms          229285 non-null  float64
 7   tenure               260604 non-null  object 
 8   propertyType         265817 non-null  object 
 9   currentEnergyRating  209511 non-null  object 
 10  sale_month           266325 non-null  int64  
 11  sale_year            266325 non-null  int64  
 12  price                266325 non-null  int64  
dtypes: float64(6), int64(3), object(4)
memory usage: 26.4+ MB


In [102]:
df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16547 entries, 0 to 16546
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   outcode              16547 non-null  object 
 1   latitude             16547 non-null  float64
 2   longitude            16547 non-null  float64
 3   bathrooms            13923 non-null  float64
 4   bedrooms             15172 non-null  float64
 5   floorAreaSqM         14541 non-null  float64
 6   livingRooms          14452 non-null  float64
 7   tenure               15957 non-null  object 
 8   propertyType         16380 non-null  object 
 9   currentEnergyRating  15050 non-null  object 
 10  sale_month           16547 non-null  int64  
 11  sale_year            16547 non-null  int64  
dtypes: float64(6), int64(2), object(4)
memory usage: 1.5+ MB


In [103]:
df_test.isna().sum()

,0
outcode,0
latitude,0
longitude,0
bathrooms,2624
bedrooms,1375
floorAreaSqM,2006
livingRooms,2095
tenure,590
propertyType,167
currentEnergyRating,1497


### Handling Missing Values - `bathrooms`

We examine the distribution of `bathrooms` and impute missing values using random sampling based on the observed distribution. This is done for both training and testing datasets.

In [104]:
df_train.isna().sum()

,0
outcode,0
latitude,0
longitude,0
bathrooms,48479
bedrooms,24843
floorAreaSqM,13806
livingRooms,37040
tenure,5721
propertyType,508
currentEnergyRating,56814


In [105]:
df_train['bathrooms'].value_counts()

,count
bathrooms,
1.0,143432
2.0,58789
3.0,11786
4.0,2662
5.0,706
6.0,321
7.0,103
8.0,33
9.0,14


In [106]:
df_test['bathrooms'].value_counts()

,count
bathrooms,
1.0,8959
2.0,3944
3.0,761
4.0,170
5.0,57
6.0,23
7.0,5
8.0,3
9.0,1


Missing values in `bathrooms` have been imputed.

In [107]:
df_train.loc[df_train['bathrooms'].isna(), 'bathrooms'] = np.random.choice(a = [1.0, 2.0, 3.0, 4.0], size = df_train['bathrooms'].isna().sum(), p = [0.5, 0.3, 0.1, 0.1])
df_test.loc[df_test['bathrooms'].isna(), 'bathrooms'] = np.random.choice(a = [1.0, 2.0, 3.0, 4.0], size = df_test['bathrooms'].isna().sum(), p = [0.5, 0.3, 0.1, 0.1])

### Handling Missing Values - `bedrooms`

Similarly, we examine the `bedrooms` distribution and impute missing values using random sampling.

In [108]:
df_train['bedrooms'].value_counts()

,count
bedrooms,
2.0,90652
3.0,61396
1.0,46794
4.0,27638
5.0,11222
6.0,2940
7.0,595
8.0,185
9.0,60


In [109]:
df_test['bedrooms'].value_counts()

,count
bedrooms,
2.0,5562
3.0,3996
1.0,2863
4.0,1803
5.0,686
6.0,194
7.0,50
8.0,13
9.0,5


Missing values in `bedrooms` have been imputed.

In [110]:
df_train.loc[df_train['bedrooms'].isna(), 'bedrooms'] = np.random.choice(a = [2.0, 3.0, 1.0, 4.0], size = df_train['bedrooms'].isna().sum(), p = [0.5, 0.3, 0.1, 0.1])
df_test.loc[df_test['bedrooms'].isna(), 'bedrooms'] = np.random.choice(a = [2.0, 3.0, 1.0, 4.0], size = df_test['bedrooms'].isna().sum(), p = [0.5, 0.3, 0.1, 0.1])

### Handling Missing Values - `floorAreaSqM` and `livingRooms`

For `floorAreaSqM` and `livingRooms`, we first create indicator columns to mark where values were originally missing. Then, we impute `livingRooms` with the most frequent value (1.0). For `floorAreaSqM`, we impute missing values based on the median `floorAreaSqM` for each `livingRooms` group. If there are still missing values (e.g., if a `livingRooms` category had no non-null `floorAreaSqM`), we fill them with the overall median `floorAreaSqM`.

In [111]:
df_train['floorAreaSqM'].value_counts()

,count
floorAreaSqM,
55.0,4477
70.0,3911
56.0,3761
80.0,3512
54.0,3454
...,...
490.0,2
469.0,1
421.0,1


In [112]:
df_test['floorAreaSqM'].value_counts()

,count
floorAreaSqM,
55.0,286
70.0,238
54.0,220
56.0,218
80.0,209
...,...
308.0,1
448.0,1
403.0,1


In [113]:
df_test['floorAreaSqM'].value_counts()

,count
floorAreaSqM,
55.0,286
70.0,238
54.0,220
56.0,218
80.0,209
...,...
308.0,1
448.0,1
403.0,1


In [114]:
df_train['floorAreaSqM'].value_counts()

,count
floorAreaSqM,
55.0,4477
70.0,3911
56.0,3761
80.0,3512
54.0,3454
...,...
490.0,2
469.0,1
421.0,1


In [115]:
len(df_train['floorAreaSqM'])

266325

In [116]:
df_train['livingRooms'].value_counts()

,count
livingRooms,
1.0,174397
2.0,45182
3.0,7891
4.0,1380
5.0,328
6.0,76
7.0,26
8.0,4
9.0,1


In [117]:
len(df_train['livingRooms'])

266325

In [118]:
df_test.isna().sum()

,0
outcode,0
latitude,0
longitude,0
bathrooms,0
bedrooms,0
floorAreaSqM,2006
livingRooms,2095
tenure,590
propertyType,167
currentEnergyRating,1497


In [119]:
df_train.isna().sum()

,0
outcode,0
latitude,0
longitude,0
bathrooms,0
bedrooms,0
floorAreaSqM,13806
livingRooms,37040
tenure,5721
propertyType,508
currentEnergyRating,56814


Creating indicator columns for missing `livingRooms` and `floorAreaSqM`.

In [120]:
df_train['rooms_is_missing'] = df_train['livingRooms'].isna().astype(int)
df_train['area_is_missing'] = df_train['floorAreaSqM'].isna().astype(int)

Imputing missing `livingRooms` with 1.0 and `floorAreaSqM` with median grouped by `livingRooms` for the training set.

In [121]:
df_train['livingRooms'] = df_train['livingRooms'].fillna(1.0)

df_train['floorAreaSqM'] = df_train['floorAreaSqM'].fillna(df_train.groupby('livingRooms')['floorAreaSqM'].transform('median'))

Applying similar imputation steps for the test set.

In [122]:
df_test['rooms_is_missing'] = df_test['livingRooms'].isna().astype(int)
df_test['area_is_missing'] = df_test['floorAreaSqM'].isna().astype(int)

In [123]:
df_test['livingRooms'] = df_test['livingRooms'].fillna(1.0)

In [124]:
df_test['floorAreaSqM'] = df_test['floorAreaSqM'].fillna(df_test.groupby('livingRooms')['floorAreaSqM'].transform('median'))

Final imputation for any remaining `floorAreaSqM` missing values in the training set using the overall median.

In [125]:
print(df_train[df_train['floorAreaSqM'].isna()]['livingRooms'])

4925      8.0
19711     8.0
43856     8.0
182691    9.0
258245    8.0
Name: livingRooms, dtype: float64


In [126]:
df_train['floorAreaSqM'] = df_train['floorAreaSqM'].fillna(df_train['floorAreaSqM'].median())

### Handling Missing Values - `tenure`

We examine the `tenure` distribution and impute missing values by randomly sampling 'Leasehold' or 'Freehold' based on their proportions.

In [127]:
df_train['propertyType'].value_counts()

,count
propertyType,
Purpose Built Flat,68726
Flat/Maisonette,61139
Mid Terrace House,45649
Converted Flat,32552
Semi-Detached House,20475
Terrace Property,15114
End Terrace House,13063
Detached House,6666
Terraced,927


In [128]:
len(df_train['tenure'])

266325

In [129]:
df_test['tenure'].value_counts()

,count
tenure,
Leasehold,9017
Freehold,6643
Feudal,229
Shared,68


Missing values in `tenure` have been imputed.

In [130]:
df_train.loc[df_train['tenure'].isna(), 'tenure'] = np.random.choice(a = ['Leasehold', 'Freehold'], size = df_train['tenure'].isna().sum(), p = [0.7, 0.3])
df_test.loc[df_test['tenure'].isna(), 'tenure'] = np.random.choice(a = ['Leasehold', 'Freehold'], size = df_test['tenure'].isna().sum(), p = [0.7, 0.3])

In [131]:
df_train['tenure'].value_counts()

,count
tenure,
Leasehold,159933
Freehold,103794
Feudal,1906
Shared,692


### Handling Missing Values - `propertyType`

Missing values in `propertyType` are imputed with 'Purpose Built Flat', as it is a common type.

In [132]:
df_train['propertyType'] = df_train['propertyType'].fillna('Purpose Built Flat')
df_test['propertyType'] = df_test['propertyType'].fillna('Purpose Built Flat')

### Handling Missing Values - `currentEnergyRating`

We create an indicator column for missing `currentEnergyRating` values, impute missing values with 'D' (the most frequent rating), and then map the categorical ratings to numerical values.

In [133]:
df_train['currentEnergyRating'].value_counts()

,count
currentEnergyRating,
D,87925
C,78356
B,20836
E,20253
F,1519
G,436
A,186


In [134]:
df_test['currentEnergyRating'].value_counts()

,count
currentEnergyRating,
D,5829
C,5596
B,2037
E,1374
F,127
G,67
A,20


In [135]:
df_test.isna().sum()

,0
outcode,0
latitude,0
longitude,0
bathrooms,0
bedrooms,0
floorAreaSqM,0
livingRooms,0
tenure,0
propertyType,0
currentEnergyRating,1497


Creating indicator column for missing `currentEnergyRating` and imputing for the training set.

In [136]:
df_train['energy_is_missing'] = df_train['currentEnergyRating'].isna().astype(int)

In [137]:
df_train['currentEnergyRating'] = df_train['currentEnergyRating'].fillna('D')

Defining a dictionary to map energy ratings to numerical values.

In [138]:
rating_dict = {'A': 7, 'B': 6, 'C': 5, 'D': 4, 'E': 3, 'F': 2, 'G': 1}

Mapping energy ratings to numerical values for the training set.

In [139]:
df_train['currentEnergyRating'] = df_train['currentEnergyRating'].map(rating_dict)

Applying the same steps for the test set.

In [140]:
df_test['energy_is_missing'] = df_test['currentEnergyRating'].isna().astype(int)
df_test['currentEnergyRating'] = df_test['currentEnergyRating'].fillna('D')
df_test['currentEnergyRating'] = df_test['currentEnergyRating'].map(rating_dict)

### Verifying Data Cleaning

We check `df.info()` again to ensure all missing values have been handled and data types are appropriate.

In [141]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 266325 entries, 0 to 266324
Data columns (total 16 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   outcode              266325 non-null  object 
 1   latitude             266325 non-null  float64
 2   longitude            266325 non-null  float64
 3   bathrooms            266325 non-null  float64
 4   bedrooms             266325 non-null  float64
 5   floorAreaSqM         266325 non-null  float64
 6   livingRooms          266325 non-null  float64
 7   tenure               266325 non-null  object 
 8   propertyType         266325 non-null  object 
 9   currentEnergyRating  266325 non-null  int64  
 10  sale_month           266325 non-null  int64  
 11  sale_year            266325 non-null  int64  
 12  price                266325 non-null  int64  
 13  rooms_is_missing     266325 non-null  int64  
 14  area_is_missing      266325 non-null  int64  
 15  energy_is_missing

In [142]:
df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16547 entries, 0 to 16546
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   outcode              16547 non-null  object 
 1   latitude             16547 non-null  float64
 2   longitude            16547 non-null  float64
 3   bathrooms            16547 non-null  float64
 4   bedrooms             16547 non-null  float64
 5   floorAreaSqM         16547 non-null  float64
 6   livingRooms          16547 non-null  float64
 7   tenure               16547 non-null  object 
 8   propertyType         16547 non-null  object 
 9   currentEnergyRating  16547 non-null  int64  
 10  sale_month           16547 non-null  int64  
 11  sale_year            16547 non-null  int64  
 12  rooms_is_missing     16547 non-null  int64  
 13  area_is_missing      16547 non-null  int64  
 14  energy_is_missing    16547 non-null  int64  
dtypes: float64(6), int64(6), object(3)
m

## 3. Feature Engineering

We use one-hot encoding for categorical features like `outcode`, `tenure`, and `propertyType` to convert them into a numerical format suitable for machine learning models.

In [143]:
df_train['outcode'].value_counts()

,count
outcode,
SE18,4444
SW2,4440
N16,4163
SW4,4102
SW16,4006
...,...
EC3V,17
EC2R,7
EC2V,7


In [144]:
df_train = pd.get_dummies(df_train, columns=['outcode', 'tenure', 'propertyType'], drop_first=True, dtype=int)
df_test = pd.get_dummies(df_test, columns=['outcode', 'tenure', 'propertyType'], drop_first=True, dtype=int)

Aligning columns between `df_train` and `df_test` after one-hot encoding to ensure they have the same features. `fill_value=0` handles columns present in one but not the other.

In [145]:
df_train, df_test = df_train.align(df_test, join='left', axis=1, fill_value=0)

## 4. Model Preparation

We import necessary libraries for scaling and model building. Then, we separate the target variable (`price`) from the features in the training set and prepare the test set.

In [146]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error


Splitting data into training features (X_train), training target (y_train), and test features (X_test).

In [147]:
print(df_train.columns.tolist())

['latitude', 'longitude', 'bathrooms', 'bedrooms', 'floorAreaSqM', 'livingRooms', 'currentEnergyRating', 'sale_month', 'sale_year', 'price', 'rooms_is_missing', 'area_is_missing', 'energy_is_missing', 'outcode_E10', 'outcode_E11', 'outcode_E12', 'outcode_E13', 'outcode_E14', 'outcode_E15', 'outcode_E16', 'outcode_E17', 'outcode_E18', 'outcode_E1W', 'outcode_E2', 'outcode_E3', 'outcode_E4', 'outcode_E5', 'outcode_E6', 'outcode_E7', 'outcode_E8', 'outcode_E9', 'outcode_EC1A', 'outcode_EC1M', 'outcode_EC1N', 'outcode_EC1R', 'outcode_EC1V', 'outcode_EC1Y', 'outcode_EC2A', 'outcode_EC2M', 'outcode_EC2R', 'outcode_EC2V', 'outcode_EC2Y', 'outcode_EC3A', 'outcode_EC3M', 'outcode_EC3N', 'outcode_EC3R', 'outcode_EC3V', 'outcode_EC4A', 'outcode_EC4M', 'outcode_EC4R', 'outcode_EC4V', 'outcode_EC4Y', 'outcode_N1', 'outcode_N10', 'outcode_N11', 'outcode_N12', 'outcode_N13', 'outcode_N14', 'outcode_N15', 'outcode_N16', 'outcode_N17', 'outcode_N18', 'outcode_N19', 'outcode_N2', 'outcode_N20', 'outcode

In [148]:
print(df_test.columns.tolist())

['latitude', 'longitude', 'bathrooms', 'bedrooms', 'floorAreaSqM', 'livingRooms', 'currentEnergyRating', 'sale_month', 'sale_year', 'price', 'rooms_is_missing', 'area_is_missing', 'energy_is_missing', 'outcode_E10', 'outcode_E11', 'outcode_E12', 'outcode_E13', 'outcode_E14', 'outcode_E15', 'outcode_E16', 'outcode_E17', 'outcode_E18', 'outcode_E1W', 'outcode_E2', 'outcode_E3', 'outcode_E4', 'outcode_E5', 'outcode_E6', 'outcode_E7', 'outcode_E8', 'outcode_E9', 'outcode_EC1A', 'outcode_EC1M', 'outcode_EC1N', 'outcode_EC1R', 'outcode_EC1V', 'outcode_EC1Y', 'outcode_EC2A', 'outcode_EC2M', 'outcode_EC2R', 'outcode_EC2V', 'outcode_EC2Y', 'outcode_EC3A', 'outcode_EC3M', 'outcode_EC3N', 'outcode_EC3R', 'outcode_EC3V', 'outcode_EC4A', 'outcode_EC4M', 'outcode_EC4R', 'outcode_EC4V', 'outcode_EC4Y', 'outcode_N1', 'outcode_N10', 'outcode_N11', 'outcode_N12', 'outcode_N13', 'outcode_N14', 'outcode_N15', 'outcode_N16', 'outcode_N17', 'outcode_N18', 'outcode_N19', 'outcode_N2', 'outcode_N20', 'outcode

In [149]:
X_train = df_train.drop('price', axis=1)
y_train = df_train['price']
X_test = df_test.copy()

## 5. Model Training

### Model Definition: HistGradientBoostingRegressor

We are using `HistGradientBoostingRegressor`, an ensemble tree-based model known for its high performance and speed on tabular data. We configure it with `max_iter=150` for the number of boosting stages and `random_state=42` for reproducibility. The `verbose=1` parameter provides output during training.

In [150]:
tree_model = HistGradientBoostingRegressor(max_iter=150, random_state=42, verbose=1)

In [151]:
tree_model.fit(X_train, y_train)

Binning 0.382 GB of training data: 2.538 s
Binning 0.042 GB of validation data: 0.106 s
Fitting gradient boosted rounds:
Fit 150 trees in 65.973 s, (4650 total leaves)
Time spent computing histograms: 59.353s
Time spent finding best splits:  0.298s
Time spent applying splits:      1.125s
Time spent predicting:           0.116s


HistGradientBoostingRegressor(max_iter=150, random_state=42, verbose=1)

In [92]:
y_train_pred_tree = tree_model.predict(X_train)

## 6. Model Evaluation
  After training, we evaluate the model's performance using metrics like Root Mean Squared Error (RMSE) to understand how well it predicts house prices. A lower RMSE indicates a better fit of the model to the data.

In [93]:
rmse_tree = np.sqrt(mean_squared_error(y_train, y_train_pred_tree))
mae_tree = mean_absolute_error(y_train, y_train_pred_tree)

In [154]:
print(f"RMSE = {rmse_tree:.2f}")
print(f"MAE = {mae_tree:.2f}")
print(f"Min = {y_train_pred_tree.min():.2f}")

RMSE = 795380.60
MAE = 175556.17
Min = -227321.73


#### Model Evaluation after Prediction Clipping

Since predictions can sometimes be negative or unrealistically low for housing prices, we clip the predicted values. We set a minimum price of 10000 to ensure more realistic predictions. We then recalculate the RMSE and MAE metrics for these adjusted predictions.

In [155]:
y_train_pred_clipped = np.clip(y_train_pred_tree, a_min=10000, a_max=None)

In [156]:
rmse_clipped = np.sqrt(mean_squared_error(y_train, y_train_pred_clipped))
mae_clipped = mean_absolute_error(y_train, y_train_pred_clipped)

In [158]:
print(f"RMSE = {rmse_clipped:.2f}")
print(f"MAE = {mae_clipped:.2f}")
print(f"Min = {y_train_pred_clipped.min():.2f}")

RMSE = 795376.42
MAE = 175546.41
Min = 10000.00


In [159]:
X_test = X_test.drop('price', axis=1)

## 7. Prediction and Submission

After training the model, we use it to make predictions on the test set and prepare the submission file.

In [162]:
y_pred = tree_model.predict(X_test)

Assigning the predicted prices to the `price` column in the submission DataFrame.

In [163]:
df_sub['price'] = y_pred

Displaying the first few rows of the submission DataFrame.

In [164]:
df_sub

,ID,price
0,266325,507882.504703
1,266326,476211.112266
2,266327,478907.352343
3,266328,503005.794181
4,266329,495223.598435
...,...,...
16542,282867,396852.477397
16543,282868,396852.477397
16544,282869,466147.382089
16545,282870,888067.658023


Saving the submission DataFrame to a CSV file named 'sub.csv'.

In [166]:
df_sub.to_csv('sub.csv', index=False, index_label = False)

## Conclusion

This notebook demonstrates a complete workflow for predicting London housing prices, from data loading and extensive preprocessing to model training with `HistGradientBoostingRegressor`, evaluation, and submission generation. The steps include handling missing values, feature engineering using one-hot encoding, and a robust evaluation process with prediction clipping.